## Prepare ADRD dataset

In [1]:
## Load packages ----
import numpy as np
import pandas as pd
import sshtunnel
import psycopg2 as pg
import os
import feather

import json
import sys

import seaborn as sns
import matplotlib.pyplot as plt

/n/home_fasse/maudirac/.conda/envs/medicare_QC/lib/python3.9/site-packages/paramiko/transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [2]:
## read zip to county crosswalk ----
zip_to_county = pd.read_csv('../data/input/remote/zip_county_2010.csv')
zip_to_county = zip_to_county[['ZIP', 'COUNTY']]
zip_to_county = zip_to_county.rename(columns = {'ZIP':'zip', 'COUNTY':'county'})
zip_w = zip_to_county.groupby(['zip'])['county'].count().reset_index()
zip_w = zip_w.rename(columns = {'county':'w'})
zip_w['w'] = 1 / zip_w.w
zip_to_county = zip_to_county.merge(zip_w)
zip_to_county.w.describe()

count    46875.000000
mean         0.775168
std          0.278345
min          0.166667
25%          0.500000
50%          1.000000
75%          1.000000
max          1.000000
Name: w, dtype: float64

In [3]:
## Open ssh tunnel to DB host ----
tunnel = sshtunnel.SSHTunnelForwarder(
    ('nsaph.rc.fas.harvard.edu', 22),
    ssh_username=f'{os.environ["MY_NSAPH_SSH_USERNAME"]}',
    ssh_private_key=f'{os.environ["HOME"]}/.ssh/id_rsa', 
    ssh_password=f'{os.environ["MY_NSAPH_SSH_PASSWORD"]}', 
    remote_bind_address=("localhost", 5432)
)

tunnel.start()

## Open connection to DB ----
connection = pg.connect(
    host='localhost',
    database='nsaph2',
    user=f'{os.environ["MY_NSAPH_DB_USERNAME"]}',
    password=f'{os.environ["MY_NSAPH_DB_PASSWORD"]}', 
    port=tunnel.local_bind_port
)

## Beneficiary counts

In [5]:
## year range of interest ----
years_ = [y_ for y_ in range(2000, 2019)]
years_.remove(2015) # not available in DORIEH as of Jan 2023 

## obtain beneficiary counts per zipcode ----

bene_zip_list = list()

for y_ in years_: 
    print(y_)
    
    ## Define query ----
    sql_query = f"""
    SELECT 
        zip,
        year,
        race, 
        sex,
        case 
            when age < 65 then '<65'
            when age >= 65 and age < 75 then '[65,75)'
            when age >= 75 and age < 85 then '[75,85)'
            when age >= 85 then '>85'
        end age_grp,
        count(*) as n_enrollees
    FROM 
        medicare.enrollments
        LEFT JOIN medicare.beneficiaries ON medicare.enrollments.bene_id = medicare.beneficiaries.bene_id
    WHERE 
        year in ('{y_}') AND 
        race in ('1', '2') AND
        sex in ('1', '2')
    GROUP BY 
        age_grp, 
        year, 
        zip, 
        race, 
        sex
    ;
    """
    ## Request query ----
    %time b = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()
    bene_zip_list.append(b)

2000
CPU times: user 3.58 s, sys: 932 ms, total: 4.51 s
Wall time: 5min 37s
2001
CPU times: user 3.19 s, sys: 969 ms, total: 4.16 s
Wall time: 56.9 s
2002
CPU times: user 3.06 s, sys: 945 ms, total: 4 s
Wall time: 57.3 s
2003
CPU times: user 2.52 s, sys: 778 ms, total: 3.3 s
Wall time: 53.9 s
2004
CPU times: user 2.75 s, sys: 806 ms, total: 3.55 s
Wall time: 54.3 s
2005
CPU times: user 2.77 s, sys: 771 ms, total: 3.54 s
Wall time: 55.2 s
2006
CPU times: user 2.77 s, sys: 810 ms, total: 3.58 s
Wall time: 1min
2007
CPU times: user 2.83 s, sys: 898 ms, total: 3.73 s
Wall time: 10min 33s
2008
CPU times: user 2.69 s, sys: 952 ms, total: 3.64 s
Wall time: 5min 52s
2009
CPU times: user 3.07 s, sys: 916 ms, total: 3.98 s
Wall time: 18min 38s
2010
CPU times: user 2.79 s, sys: 888 ms, total: 3.68 s
Wall time: 12min 7s
2011
CPU times: user 2.92 s, sys: 996 ms, total: 3.91 s
Wall time: 19min 36s
2012
CPU times: user 2.97 s, sys: 953 ms, total: 3.92 s
Wall time: 21min 33s
2013
CPU times: user 2.9 s

In [6]:
## crosswalk to counties ----
bene_zip_df = pd.concat(bene_zip_list)
bene_county_df = bene_zip_df.merge(zip_to_county)
bene_county_df['n_enrollees'] = bene_county_df.n_enrollees * bene_county_df.w
bene_county_df = bene_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'])['n_enrollees'].sum().reset_index()

In [7]:
## total number of enrollees in zipcodes ----
bene_zip_df.n_enrollees.sum()

819584737

In [8]:
## total number of enrollees in counties ----
bene_county_df.n_enrollees.sum()

792564236.0000001

In [11]:
## save adrd_county_df
#bene_county_df.to_csv("../data/input/local/bene_county_df.csv", index=False)
bene_county_df.to_feather("../data/input/local/bene_county_df.feather")